# 19 - Analogo en R (recreado desde cero)

## Tema base

**19 - Metodos magicos: disenar objetos que se comportan como tipos nativos**


## Objetivos de la sesion en R

1. Mantener teoria del notebook Python en equivalente R.
2. Usar codigo 100% R ejecutable.
3. Conservar ejercicios adaptados a R.
4. Integrar extra de probabilidad y estadistica.


## Ruta Teoria-Codigo (alternada)

Se organiza la sesion en bloques cortos que alternan concepto y practica.


### Bloque teorico 1

### Teoria 1

## 1. Problema que resuelven los metodos magicos

Una clase puede "funcionar" con metodos normales, pero sentirse torpe al usarla.
Ejemplos de friccion:

- imprimir objetos devuelve algo poco util,
- no puedes usar `len(obj)` aunque el objeto tiene elementos,
- no puedes comparar, ordenar ni sumar objetos de forma natural,
- no puedes integrarlos bien con `for`, `in`, `sorted`, `set`, etc.

Los metodos magicos conectan tu clase con el ecosistema de R.
La meta no es "usar dunder por moda", sino ofrecer una API que se lea natural.

### Teoria 2

## 2. Modelo mental: R invoca protocolos, no "magia"

Cuando escribes `len(x)`, R llama internamente `x.__len__()`.
Cuando escribes `a + b`, R intenta `a.__add__(b)`.
Cuando escribes `for item in x`, busca un protocolo de iteracion.

Puntos clave:

1. No llamas estos metodos de forma directa en uso normal.
2. Cada dunder representa un contrato de comportamiento.
3. Si implementas el contrato a medias, creas bugs sutiles.

Por eso conviene pensar primero la semantica de dominio y luego codificar.

### Teoria 3

## 3. `__repr__` vs `__str__`: depurar bien y comunicar mejor

`__repr__`:

- orientado a desarrolladores,
- debe ser claro, sin ambiguedad,
- idealmente util para reconstruir estado.

`__str__`:

- orientado a usuarios finales,
- mas legible y narrativo,
- puede omitir detalles tecnicos.

Regla practica: implementa al menos `__repr__`. Si hay interfaz humana, agrega `__str__`.

### Teoria 4

## 4. `__len__` y `__bool__`: verdad logica consistente

R evalua un objeto como falso si:

1. define `__bool__` y retorna `False`, o
2. no define `__bool__`, pero `__len__` retorna 0.

Diseno recomendado:

- usa `__len__` cuando modelas una coleccion,
- define `__bool__` solo si tu nocion de "vacio" no depende de longitud,
- evita reglas sorpresivas (ejemplo: objeto con elementos que evalua False).

### Teoria 5

## 5. `__eq__` y `__hash__`: igualdad correcta sin romper estructuras hash

Si dos objetos representan la misma identidad de dominio, `==` deberia reflejarlo.
Pero cuidado: si redefiniste `__eq__`, R puede deshabilitar `__hash__` para protegerte.

Contrato importante:

1. Si `a == b`, entonces `hash(a) == hash(b)`.
2. Objetos usados como clave deben ser inmutables en los campos que participan en hash.
3. Si no hay inmutabilidad clara, evita `__hash__`.

### Teoria 6

## 6. Comparaciones y orden: `__lt__` con criterio de negocio

`sorted()` y `list.sort()` necesitan poder comparar elementos.
Si tu clase define orden, ese orden debe ser coherente y explicable.

Recomendaciones:

- define el criterio principal de forma explicita,
- para comparaciones no soportadas, retorna `NotImplemented`,
- no mezcles criterios incompatibles sin documentarlos.

### Teoria 7

## 7. Operadores: `__add__`, `__radd__`, `__iadd__` con semantica clara

Sobrecargar operadores puede mejorar legibilidad o arruinarla.

Buenas practicas:

1. Solo sobrecarga si el operador tiene sentido natural en tu dominio.
2. Si no sabes combinar tipos distintos, retorna `NotImplemented`.
3. Define si la operacion crea objeto nuevo (`__add__`) o muta (`__iadd__`).
4. Evita "magia oculta" que sorprenda al equipo.


In [ ]:
moneda <- function(valor, divisa='MXN'){structure(list(valor=as.numeric(valor), divisa=divisa), class='moneda')}
print.moneda <- function(x,...){cat(sprintf('%.2f %s\n', x$valor, x$divisa))}
Ops.moneda <- function(e1,e2){op <- .Generic; if(!inherits(e1,'moneda')||!inherits(e2,'moneda')) stop('ambos operandos deben ser moneda'); if(e1$divisa!=e2$divisa) stop('divisas incompatibles'); if(op=='+') return(moneda(e1$valor+e2$valor,e1$divisa)); if(op=='-') return(moneda(e1$valor-e2$valor,e1$divisa)); stop('operador no soportado')}


### Bloque teorico 2

### Teoria 8

## 8. Clases que se sienten como colecciones

Con estos metodos puedes integrarte al ecosistema de colecciones:

- `__getitem__`: acceso por indice o llave,
- `__setitem__`: asignacion por indice o llave,
- `__contains__`: operador `in`,
- `__iter__`: iteracion con `for`.

Si tu clase representa datos indexables, vale mucho la pena implementar este protocolo.

### Teoria 9

## 9. `__call__`: objetos configurables que actuan como funciones

`__call__` permite que una instancia sea invocable con parentesis.
Es util cuando quieres combinar:

- configuracion inicial,
- estado acumulado,
- y una interfaz simple de uso.

Piensa en validadores, filtros, politicas de negocio o pipelines.

### Teoria 10

## 10. Caso real pequeno: clase de dominio con comportamiento natural

Escenario: quieres modelar rangos horarios para una agenda.
Necesitas comparar, imprimir y validar pertenencia de minutos.

Decision de diseno:

1. `__contains__` para `minuto in rango`.
2. `__len__` para saber duracion.
3. `__repr__` para depurar incidencias.

La clase queda facil de leer para cualquier persona del equipo.

### Teoria 11

## 11. Errores comunes con metodos magicos

1. Implementar dunder "porque se ve avanzado", sin necesidad real.
2. Definir `__eq__` y olvidar `__hash__` cuando la clase ira a `set` o `dict`.
3. Sobrecargar operadores con semantica confusa (ejemplo: `+` que borra datos).
4. Romper expectativas de verdad logica (`bool(obj)`) respecto a su estado.
5. Hacer comparaciones parciales que no son transitivas.
6. Capturar errores y devolver valores silenciosos que ocultan bugs.
7. No cubrir contratos con pruebas de comportamiento.

### Teoria 12

## 12. Checklist rapido de diseno

Antes de agregar un metodo magico, responde:

1. Que operacion de R quiero habilitar y por que?
2. Mi implementacion respeta expectativas de tipos nativos?
3. La semantica esta alineada con el dominio del problema?
4. Este metodo introduce ambiguedad para el equipo?
5. Tengo pruebas que validen casos normales y bordes?

Si no puedes justificarlo, probablemente no necesitas ese dunder.

### Teoria 13

## 14. Resumen de conceptos clave

1. Los metodos magicos conectan tu clase con operaciones nativas de R.
2. Cada dunder debe responder a una necesidad real del dominio.
3. `__repr__` mejora depuracion; `__str__` mejora comunicacion con usuarios.
4. Igualdad y hash exigen consistencia e idealmente inmutabilidad.
5. Orden y operadores deben ser predecibles y defendibles.
6. Protocolos de coleccion (`len`, iteracion, indexado, pertenencia) reducen friccion de uso.
7. `__call__` permite objetos configurables con interfaz simple.
8. Buen diseno se demuestra en claridad de API, pruebas y facilidad de evolucion.


In [ ]:
a <- moneda(150); b <- moneda(30)
a+b
a-b
tryCatch(moneda(100,'USD') + moneda(200,'MXN'), error=function(e) e$message)


## Extra de probabilidad y estadistica

Simulacion de retornos y resumen de riesgo/retorno.


In [ ]:
set.seed(42)
ret <- rnorm(250, mean=0.0008, sd=0.02)
capital <- 100*cumprod(1+ret)
cat('retorno medio=', mean(ret), ' volatilidad=', sd(ret), '\n')
plot(capital, type='l', col='darkgreen')


## Analogias R y Python

- `Ops.clase` en R es an?logo a metodos especiales de operadores en Python.


In [ ]:
# Checklist rapido R vs Python
# 1) ?Que cambia en sintaxis?
# 2) ?Que cambia en estructura de datos?
# 3) ?Que cambia en manejo de NA/errores?


## Errores comunes

- Sobrecargar operadores sin reglas claras.
- Mezclar divisas sin validacion.
- No documentar semantica de operaciones.


In [ ]:
# Auto-verificacion de errores comunes
# Define un caso borde y valida que tu solucion en R no falle silenciosamente.


## Ejercicios para pensar (no copiar codigo)

Primero argumenta en texto. Luego, si hace falta, valida con experimentos en R.


### Ejercicio reflexivo 1

### Ejercicio 1

# 19 - Metodos magicos: disenar objetos que se comportan como tipos nativos

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que son los metodos magicos (`__dunder__`) y cuando conviene implementarlos.
2. Diferenciar `__repr__` y `__str__` para depuracion vs experiencia de usuario.
3. Disenar semantica consistente para verdad logica (`__bool__`) y longitud (`__len__`).
4. Implementar igualdad (`__eq__`) y hash (`__hash__`) sin romper diccionarios ni sets.
5. Definir orden (`__lt__`, `__le__`, etc.) con criterio de dominio y pruebas.
6. Sobrecargar operadores (`__add__`, `__radd__`, `__iadd__`) evitando sorpresas.
7. Exponer colecciones personalizadas con `__getitem__`, `__iter__` y `__contains__`.
8. Usar `__call__` para objetos configurables que actuan como funciones.
9. Identificar errores frecuentes al modelar el protocolo de datos de R.
10. Practicar decisiones de diseno en ejercicios que priorizan razonamiento tecnico.


### Ejercicio reflexivo 2

### Ejercicio 2

## 13. Ejercicios de pensamiento y practica real

La meta no es copiar codigo.
La meta es tomar decisiones de diseno con argumentos tecnicos y validarlas con pruebas.

Recomendacion: escribe primero supuestos e invariantes, despues implementa.


### Ejercicio reflexivo 3

### Ejercicio 3

### Ejercicio 1: contrato minimo para un tipo de dominio

**Escenario**: sistema academico con una clase `Calificacion`.

**Tarea**:
1. Decide que metodos magicos SI deberia tener (`__repr__`, `__str__`, `__eq__`, otros).
2. Justifica cuales NO deberia tener y por que.
3. Define dos invariantes del dominio.
4. Implementa una version minima coherente.


### Ejercicio reflexivo 4

### Ejercicio 4

### Ejercicio 2: `__repr__` util para depuracion real

Analiza esta clase y mejora su representacion.

**Tarea**:
1. Detecta por que el `repr` actual dificulta debug.
2. Reescribe `__repr__` con informacion critica.
3. Agrega un `__str__` orientado a usuario.
4. Muestra un ejemplo donde la mejora ahorre tiempo de diagnostico.


### Ejercicio reflexivo 5

### Ejercicio 5

### Ejercicio 3: igualdad de identidad vs igualdad de estado

**Escenario**: clase `Empleado` con `id_empleado`, `nombre`, `salario`.

**Tarea**:
1. Elige criterio de igualdad para tu negocio y argumentalo.
2. Implementa `__eq__` y, si aplica, `__hash__`.
3. Demuestra con ejemplos como cambia el comportamiento en `set`.
4. Explica riesgos de elegir mal el criterio.


### Ejercicio reflexivo 6

### Ejercicio 6

### Ejercicio 4: bug sutil de hash por mutabilidad

La siguiente clase parece correcta, pero puede romper diccionarios.

**Tarea**:
1. Reproduce el bug.
2. Explica la causa exacta.
3. Propone dos soluciones con tradeoffs.
4. Implementa una solucion segura.


### Ejercicio reflexivo 7

### Ejercicio 7

### Ejercicio 5: orden total para una cola de prioridad

**Escenario**: tickets con severidad, impacto y tiempo de creacion.

**Tarea**:
1. Define una tupla de ordenamiento defendible.
2. Implementa comparaciones minimas (`__eq__`, `__lt__`) y usa `@total_ordering`.
3. Verifica transitividad con al menos 3 objetos.
4. Explica que pasaria si cambias el criterio en produccion.


### Ejercicio reflexivo 8

### Ejercicio 8

### Ejercicio 6: coleccion personalizada con slicing

**Escenario**: historial de transacciones con lectura por indice y por rango.

**Tarea**:
1. Implementa `__getitem__` para soportar indice y slice.
2. Define que error lanzar en indices invalidos.
3. Agrega `__len__` y `__iter__`.
4. Justifica por que tu API se parece a una lista y donde difiere.


### Ejercicio reflexivo 9

### Ejercicio 9

### Ejercicio 7: `__call__` para reglas configurables

**Escenario**: motor antifraude con reglas que pueden encadenarse.

**Tarea**:
1. Crea al menos 3 reglas invocables (`__call__`).
2. Cada regla recibe una compra y devuelve `(ok, motivo)`.
3. Dise?a un motor que combine reglas sin conocer clases concretas.
4. Argumenta como agregarias una regla nueva sin tocar el motor.


### Ejercicio reflexivo 10

### Ejercicio 10

### Ejercicio 8: sobrecarga de operadores sin ambiguedad

**Escenario**: clase `Vector2D` para calculos basicos.

**Tarea**:
1. Implementa `__add__` y `__sub__` entre vectores.
2. Decide si soportaras multiplicacion por escalar y donde (`__mul__`, `__rmul__`).
3. Retorna `NotImplemented` en tipos no compatibles.
4. Explica una decision para evitar malentendidos matematicos.


### Ejercicio reflexivo 11

### Ejercicio 11

### Ejercicio 9: diseno de API para objetos "truthy/falsy"

**Escenario**: clase `ConexionBD` con estados `conectado`, `latencia_ms`, `errores`.

**Tarea**:
1. Decide si conviene implementar `__bool__`.
2. Define criterio explicito de "conexion usable".
3. Implementa la clase y muestra ejemplos limite.
4. Explica riesgos de ocultar demasiada logica en `if conexion:`.


### Ejercicio reflexivo 12

### Ejercicio 12

### Ejercicio 10: mini proyecto integrador

**Reto**: construir un sistema de carrito para e-commerce con objetos de dominio propios.

Debe incluir:
1. `Producto` con representacion util,
2. `LineaCarrito` comparable u ordenable por criterio que definas,
3. `Carrito` iterable y con `len`,
4. reglas de descuento invocables (`__call__`),
5. soporte de suma de subtotales con operador o metodo bien justificado.

**Entrega esperada**:
- implementacion minima funcional,
- pruebas con `assert` de contratos importantes,
- explicacion de tradeoffs y deuda tecnica consciente.


### Ejercicio reflexivo 13

### Ejercicio 13

## 15. Reto final opcional

Disena un mini motor de recomendaciones de estudio con:

- `Tema` y `Recurso` como objetos de dominio,
- reglas ponderadas para priorizar recursos,
- comparacion y orden de recomendaciones,
- interfaz de evaluacion invocable con `__call__`.

Objetivo:

1. elegir metodos magicos con criterio (no por cantidad),
2. justificar cada decision de semantica,
3. demostrar con pruebas que la API es natural y estable.


### Preguntas de cierre

1. ?Que supuesto estadistico es mas fragil en esta clase?
2. ?Que evidencia minima pedirias antes de usar este enfoque en un caso real?
3. ?Como explicarias este tema a alguien no tecnico sin perder rigor?
4. ?Que cambiaria entre una implementacion en R y una en Python para produccion?
